In [1]:
from optuna import Study, Trial, create_study
from optuna.samplers import TPESampler

from pathlib import Path

from sklearn.ensemble import RandomForestClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_selection import RFE, SelectFromModel
from sklearn.linear_model import Lasso, LinearRegression, LogisticRegression
from sklearn.metrics import accuracy_score, r2_score
from sklearn.model_selection import (
    GridSearchCV,
    cross_val_score,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from xgboost import XGBClassifier

import numpy as np
import pandas as pd
import plotly.express as px
import warnings

In [2]:
# Load the dataset into a DataFrame
df = pd.read_csv("zad2_wum_data_for_students.csv", sep=";")
df

,Class,Output,Input1,Input2,Input3,Input4,Input5,Input6,Input7,Input8,...,Input391,Input392,Input393,Input394,Input395,Input396,Input397,Input398,Input399,Input400
0,0,0.800586,-0.002583,2.184037,-0.322008,1.621241,1.192444,-0.278356,-0.207366,0.735689,...,-2.140861,1.187660,0.345238,-0.844885,0.580007,-2.605781,-0.299471,0.711487,-0.753316,0.728763
1,0,2.168475,0.668637,1.373933,-0.476868,-0.724704,0.031162,-1.845921,0.784890,1.508526,...,-1.286120,-0.900044,-0.500399,-0.126421,-0.632233,-2.557419,0.056044,0.634774,-0.259835,0.106390
2,1,-1.210777,-0.681438,-0.544753,0.441346,-0.019906,-0.192135,-0.162510,-0.998777,0.686472,...,-0.391605,-0.190147,0.793746,-0.812737,-0.068228,-0.313143,2.564096,0.848355,0.180556,-1.525615
3,1,0.505678,-0.497957,0.720712,0.149120,0.019251,1.377850,0.981337,-0.846813,0.036790,...,-0.176734,-0.947351,-0.888601,1.509450,-0.501929,-0.554909,-0.104051,0.442150,-0.056644,1.447267
4,1,-10.281033,-1.178544,0.176941,1.112202,1.234189,0.999451,-0.773329,-0.811075,1.550537,...,-0.181325,0.198960,-0.697497,-0.836371,1.652071,0.974292,1.584071,-0.202352,1.362426,1.023857
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,1,-4.298039,-0.893128,2.081556,0.796121,0.436108,-0.849635,1.129482,1.432813,-0.438694,...,-0.474294,0.039143,1.808243,-0.034847,-1.314878,-1.235939,1.010456,-2.186403,-0.157829,0.738539
1996,1,-0.431692,0.048336,0.770285,-0.354350,-1.557706,-0.182954,-0.665730,0.322526,0.658221,...,1.087032,0.095401,0.301200,1.776995,-2.045261,-1.931008,-0.683551,0.000835,-0.671151,-0.945843
1997,1,-0.056681,1.404051,-0.061405,0.180448,-0.362992,0.826353,-0.066654,0.987946,-1.266302,...,1.011431,0.458901,-0.220498,0.004950,-1.928972,-1.574129,1.421012,-0.736559,-0.540174,-1.182067
1998,0,-0.983396,-0.474238,-1.288631,-0.326170,-0.275383,-0.315331,-1.225622,-0.656750,0.777151,...,-1.146012,-0.031477,-2.461869,1.037240,0.366076,-0.541171,-0.126733,-0.265069,-0.080381,0.166985


In [3]:
df_inputs = df.filter(regex="^Input")

# Extract input (X) and output (y) variables
X = df_inputs.to_numpy()
y = df[["Class", "Output"]].to_numpy()

# Extract feature names
feature_names = df_inputs.columns

# Split the dataset into train (80%) and test (20%) subsets
# Note: Stratify is applied to preserve class distribution
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y[:, 0]
)

# Aggregate output variables by their type
y_train = {"Class": y_train[:, 0], "Output": y_train[:, 1]}
y_test = {"Class": y_test[:, 0], "Output": y_test[:, 1]}

---
**Task 1.** *Building baseline models*

In [4]:
base_regressor = LinearRegression().fit(X_train, y_train["Output"])

# Evaluate the model on training and test datasets
r2_train = r2_score(y_train["Output"], base_regressor.predict(X_train))
r2_test = r2_score(y_test["Output"], base_regressor.predict(X_test))

# Estimate the model's generalizability (using a 5-fold CV)
r2_val = cross_val_score(
    base_regressor, X_train, y_train["Output"], scoring="r2", cv=5
).mean()

print(f"r2 (train) = {r2_train}")
print(f"r2 (val)   = {r2_val}")
print(f"r2 (test)  = {r2_test}")

r2 (train) = 0.6400830958578239
r2 (val)   = 0.32850808723258895
r2 (test)  = 0.3196352655070207


In [5]:
base_classifier = LogisticRegression().fit(X_train, y_train["Class"])

# Evaluate the model on training and test datasets
accuracy_train = accuracy_score(
    y_train["Class"], base_classifier.predict(X_train)
)
accuracy_test = accuracy_score(
    y_test["Class"], base_classifier.predict(X_test)
)

# Estimate the mode's generalizability (using a 5-fold CV)
accuracy_val = cross_val_score(
    base_classifier, X_train, y_train["Class"], scoring="accuracy", cv=5
).mean()

print(f"accuracy (train) = {accuracy_train}")
print(f"accuracy (val)   = {accuracy_val}")
print(f"accuracy (test)  = {accuracy_test}")

accuracy (train) = 0.735
accuracy (val)   = 0.5081249999999999
accuracy (test)  = 0.5275


Given the both models' poor relative performance during cross-validation and on the test dataset, they do not seem to generalize well.

---
**Task 2.** *More advanced classification*

> Utilize the knowledge that the class variable depends on some of the predictors, but not necessarily all.

Because of this, the classification process will be divided into three parts:

- first, the data is scaled (required for `SVC` and recommended for `LogisticRegression`, does not affect tree-based splits),
- then, the feature selector identifies the most important features,
- finally, the classifier is trained using only those selected features.

This way, the classifier learns from significantly less noisy and more relevant data.

To find the best configuration (selector, classifier and their parameters), we will use the `Optuna` library. The candidates are as follows:

- feature selectors: `SelectFromModel` and `RFE` (*Recursive Feature Elimination*),

    Both select the most important features, but they work differently:

    - `SelectFromModel` selects features based on weights learned by a model (e.g., coefficients or feature importances) and keeps those above a certain threshold,
    - `RFE` involves repeatedly training the model and removing the least significant features until their desired number is reached.

    In short, the first one is more of a one-shot selector, while the second is an iterative elimination process.

    As for their inner estimators, the options are:

    - `LogisticRegression`, due to its use of the `ElasticNet` regularizer (note the slight parameter bias towards L1 to zero out irrelevant features' weights more aggressively),
    - `RandomForestClassifier` to account for any nonlinear dependencies.

- classifiers: `SVC` and `XGBClassifier`.

    1. The `SVC` is a distance-based model that separates data geometrically (using margins). It may be the right choice if each class forms a distinct cluster in the feature space. Depending on their shape, we test different kernels (`linear`, `poly`, `rbf`) to determine which metric works best.

    2. The `XGBClassifier` is a tree-based model that handles complex, nonlinear relationships. It may adapt better to unstructured and irregular data. However, it it also prone to overfitting - we must carefully tune its parameters (e.g., limit `max_depth` and `learning_rate`) to ensure it generalizes well.

In [6]:
def suggest_estimator(trial: Trial):
    estimator = trial.suggest_categorical(
        "estimator", ["LogisticRegression", "RandomForestClassifier"]
    )

    if estimator == "LogisticRegression":
        C = trial.suggest_categorical("estimator__C", [0.01, 0.1, 1.0])
        l1_ratio = trial.suggest_float("estimator__l1_ratio", 0.2, 1.0)

        # Note: only the 'saga' solver supports 0 <= l1_ratio <= 1
        return LogisticRegression(C=C, solver="saga", l1_ratio=l1_ratio)

    else:  # RandomForestClassifier
        n_estimators = trial.suggest_int("estimator__n_estimators", 1, 100)
        max_depth = trial.suggest_int("estimator__max_depth", 1, 6)

        return RandomForestClassifier(n_estimators, max_depth=max_depth)


def suggest_selector(trial: Trial, estimator):
    selector = trial.suggest_categorical(
        "selector", ["RFE", "SelectFromModel"]
    )

    if selector == "RFE":
        n_features_to_select = trial.suggest_int(
            "selector__n_features_to_select", 1, 50
        )
        step = trial.suggest_int("selector__step", 1, 10)

        return RFE(
            estimator, n_features_to_select=n_features_to_select, step=step
        )

    else:  # SelectFromModel
        max_features = trial.suggest_int("selector__max_features", 1, 50)

        # Note: To only select based on max_features, threshold must equal -inf
        return SelectFromModel(
            estimator, threshold=-np.inf, max_features=max_features
        )

In [7]:
def suggest_classifier(trial: Trial):
    classifier = trial.suggest_categorical(
        "classifier", ["SVC", "XGBClassifier"]
    )

    if classifier == "SVC":
        C = trial.suggest_categorical("classifier__C", [0.01, 0.1, 1.0])
        kernel = trial.suggest_categorical(
            "classifier__kernel", ["linear", "poly", "rbf"]
        )

        if kernel == "poly":
            degree = trial.suggest_int("classifier__degree", 1, 3)
        else:
            degree = 3  # default, ignored by other kernels

        if kernel == "poly" or kernel == "rbf":
            gamma = trial.suggest_float("classifier__gamma", 1e-4, 1.0)
        else:
            gamma = "scale"  # default, ignored by other kernels

        return SVC(C=C, kernel=kernel, degree=degree, gamma=gamma)

    else:  # XGBClassifier
        n_estimators = trial.suggest_int("classifier__n_estimators", 1, 100)
        max_depth = trial.suggest_int("classifier__max_depth", 1, 4)
        learning_rate = trial.suggest_categorical(
            "classifier__learning_rate", [0.01, 0.05, 0.1]
        )

        return XGBClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            learning_rate=learning_rate,
            eval_metric="logloss",
        )

In [8]:
def objective(trial: Trial):
    # Build the model (as described above)
    model = Pipeline(
        [
            ("scaler", StandardScaler()),
            ("selector", suggest_selector(trial, suggest_estimator(trial))),
            ("classifier", suggest_classifier(trial)),
        ]
    )

    # Evaluate the model (using a 5-fold CV)
    return cross_val_score(
        model, X_train, y_train["Class"], scoring="accuracy", cv=5
    ).mean()


# Create a new Optuna's optimization task
study = create_study(direction="maximize", sampler=TPESampler())

# Ignore ConvergenceWarning(s)
warnings.filterwarnings("ignore", category=ConvergenceWarning)

# Search for the optimal configuration
# Note: n_jobs=-1 uses all available CPU cores
study.optimize(objective, n_trials=1000, n_jobs=-1)

[I 2026-05-20 21:40:13,005] A new study created in memory with name: no-name-f549e054-839d-4605-87de-1a3d525c1d56


[I 2026-05-20 21:40:13,776] Trial 14 finished with value: 0.50625 and parameters: {'estimator': 'RandomForestClassifier', 'estimator__n_estimators': 4, 'estimator__max_depth': 6, 'selector': 'SelectFromModel', 'selector__max_features': 40, 'classifier': 'SVC', 'classifier__C': 1.0, 'classifier__kernel': 'rbf', 'classifier__gamma': 0.2741654713206142}. Best is trial 14 with value: 0.50625.


[I 2026-05-20 21:40:14,614] Trial 6 finished with value: 0.530625 and parameters: {'estimator': 'LogisticRegression', 'estimator__C': 0.1, 'estimator__l1_ratio': 0.9070343366793725, 'selector': 'SelectFromModel', 'selector__max_features': 35, 'classifier': 'SVC', 'classifier__C': 1.0, 'classifier__kernel': 'poly', 'classifier__degree': 1, 'classifier__gamma': 0.5071023181512373}. Best is trial 6 with value: 0.530625.


[I 2026-05-20 21:40:14,976] Trial 5 finished with value: 0.5925 and parameters: {'estimator': 'LogisticRegression', 'estimator__C': 0.01, 'estimator__l1_ratio': 0.7407944828090491, 'selector': 'SelectFromModel', 'selector__max_features': 33, 'classifier': 'XGBClassifier', 'classifier__n_estimators': 42, 'classifier__max_depth': 1, 'classifier__learning_rate': 0.05}. Best is trial 5 with value: 0.5925.


[I 2026-05-20 21:40:15,081] Trial 3 finished with value: 0.5287499999999999 and parameters: {'estimator': 'LogisticRegression', 'estimator__C': 1.0, 'estimator__l1_ratio': 0.7215836222036214, 'selector': 'SelectFromModel', 'selector__max_features': 42, 'classifier': 'SVC', 'classifier__C': 1.0, 'classifier__kernel': 'poly', 'classifier__degree': 1, 'classifier__gamma': 0.8337298619822594}. Best is trial 5 with value: 0.5925.


...

[I 2026-05-21 00:21:14,401] Trial 963 finished with value: 0.83375 and parameters: {'estimator': 'RandomForestClassifier', 'estimator__n_estimators': 69, 'estimator__max_depth': 6, 'selector': 'RFE', 'selector__n_features_to_select': 12, 'selector__step': 1, 'classifier': 'SVC', 'classifier__C': 1.0, 'classifier__kernel': 'rbf', 'classifier__gamma': 0.5247189592306181}. Best is trial 983 with value: 0.8462500000000001.


[I 2026-05-21 00:21:51,275] Trial 958 finished with value: 0.8375 and parameters: {'estimator': 'RandomForestClassifier', 'estimator__n_estimators': 73, 'estimator__max_depth': 6, 'selector': 'RFE', 'selector__n_features_to_select': 12, 'selector__step': 1, 'classifier': 'SVC', 'classifier__C': 1.0, 'classifier__kernel': 'rbf', 'classifier__gamma': 0.5392426944875627}. Best is trial 983 with value: 0.8462500000000001.


[I 2026-05-21 00:21:58,970] Trial 967 finished with value: 0.8356250000000001 and parameters: {'estimator': 'RandomForestClassifier', 'estimator__n_estimators': 74, 'estimator__max_depth': 6, 'selector': 'RFE', 'selector__n_features_to_select': 9, 'selector__step': 1, 'classifier': 'SVC', 'classifier__C': 1.0, 'classifier__kernel': 'rbf', 'classifier__gamma': 0.5807854036714224}. Best is trial 983 with value: 0.8462500000000001.


[I 2026-05-21 00:22:20,847] Trial 979 finished with value: 0.8362499999999999 and parameters: {'estimator': 'RandomForestClassifier', 'estimator__n_estimators': 73, 'estimator__max_depth': 6, 'selector': 'RFE', 'selector__n_features_to_select': 12, 'selector__step': 1, 'classifier': 'SVC', 'classifier__C': 1.0, 'classifier__kernel': 'rbf', 'classifier__gamma': 0.5904712508074291}. Best is trial 983 with value: 0.8462500000000001.


[I 2026-05-21 00:22:23,344] Trial 980 finished with value: 0.843125 and parameters: {'estimator': 'RandomForestClassifier', 'estimator__n_estimators': 73, 'estimator__max_depth': 6, 'selector': 'RFE', 'selector__n_features_to_select': 13, 'selector__step': 1, 'classifier': 'SVC', 'classifier__C': 1.0, 'classifier__kernel': 'rbf', 'classifier__gamma': 0.524966117926841}. Best is trial 983 with value: 0.8462500000000001.


In [9]:
def extract_params(study: Study, prefix: str):
    # Get parameters associated with the specified prefix
    return {
        key.removeprefix(prefix): value
        for key, value in study.best_params.items()
        if key.startswith(prefix)
    }


def get_selector(study: Study):
    estimator_params = extract_params(study, "estimator__")

    if study.best_params["estimator"] == "LogisticRegression":
        estimator = LogisticRegression(solver="saga", **estimator_params)
    else:  # RandomForestClassifier
        estimator = RandomForestClassifier(**estimator_params)

    selector_params = extract_params(study, "selector__")

    if study.best_params["selector"] == "RFE":
        return RFE(estimator, **selector_params)
    else:  # SelectFromModel
        return SelectFromModel(estimator, threshold=-np.inf, **selector_params)


def get_classifier(study: Study):
    params = extract_params(study, "classifier__")

    if study.best_params["classifier"] == "SVC":
        return SVC(**params)
    else:  # XGBClassifier
        return XGBClassifier(eval_metric="logloss", **params)


# Rebuild (and retrain) the best-performing pipeline
improved_classifier = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("selector", get_selector(study)),
        ("classifier", get_classifier(study)),
    ]
).fit(X_train, y_train["Class"])

In [10]:
# Extract selector and classifier from the model
selector = improved_classifier.named_steps["selector"]
classifier = improved_classifier.named_steps["classifier"]

print(f"selector   : {selector}")
print(f"classifier : {classifier}", end="\n\n")

# Evaluate the model on training and test datasets
accuracy_train = accuracy_score(
    y_train["Class"], improved_classifier.predict(X_train)
)
accuracy_test = accuracy_score(
    y_test["Class"], improved_classifier.predict(X_test)
)

print(f"accuracy (train) = {accuracy_train}")
print(f"accuracy (val)   = {study.best_value}")
print(f"accuracy (test)  = {accuracy_test}", end="\n\n")

# Retrieve indices of selected features
selected_features = selector.get_support(indices=True)

print(f"selected {len(selected_features)} features:")
print(*feature_names[selected_features])

selector   : RFE(estimator=RandomForestClassifier(max_depth=6, n_estimators=72),
    n_features_to_select=14, step=4)
classifier : SVC(gamma=0.5207184329307573)

accuracy (train) = 0.93375
accuracy (val)   = 0.8462500000000001
accuracy (test)  = 0.8325

selected 14 features:
Input2 Input40 Input41 Input74 Input95 Input110 Input206 Input238 Input240 Input246 Input256 Input293 Input330 Input396


|         | relative improvement over the baseline model (accuracy) |
|---------|---------------------------------------------------------|
| train   | $\color{green}+27\%$                                    |
| val     | $\color{green}+67\%$                                    |
| test    | $\color{green}+58\%$                                    |

---
**Task 3.** *More advanced regression*

> Utilize the knowledge that the output variable depends on some of the predictors, but not necessarily all of them.

Similarly to the previous task, the regression process will be divided into two parts:

- first, the data is scaled,
- then, the `Lasso` regressor is trained using all features.

However, unlike in the classification task, there is no need for an additional feature selection step. The `Lasso` model uses L1 regularization, which imposes a penalty on the absolute value of each coefficient, forcing the model to reduce the coefficients of the least significant features. Most importantly, it can reduce them to exactly zero.

By doing so, `Lasso` can effectively remove irrelevant features from the model, thereby performing feature selection by itself. Scaling ensures that this regularization is applied fairly.

To find the best parameter (alpha), we will use the `GridSearchCV` function.

In [11]:
model = Pipeline([("scaler", StandardScaler()), ("regressor", Lasso())])

# Initialize the grid search with a 5-fold CV
# Note: n_jobs=-1 uses all available CPU cores
grid = GridSearchCV(
    model,
    {"regressor__alpha": [0.01, 0.1, 1.0, 2.0, 5.0]},
    scoring="r2",
    n_jobs=-1,
    cv=5,
    verbose=2,
)

# Search for the optimal configuration
grid.fit(X_train, y_train["Output"]);

Fitting 5 folds for each of 5 candidates, totalling 25 fits


[CV] END ...............................regressor__alpha=1.0; total time=   0.0s
[CV] END ...............................regressor__alpha=5.0; total time=   0.0s
[CV] END ...............................regressor__alpha=1.0; total time=   0.0s
[CV] END ...............................regressor__alpha=5.0; total time=   0.0s
[CV] END ..............................regressor__alpha=0.01; total time=   0.0s
[CV] END ...............................regressor__alpha=2.0; total time=   0.0s
[CV] END ..............................regressor__alpha=0.01; total time=   0.0s
[CV] END ...............................regressor__alpha=2.0; total time=   0.0s
[CV] END ...............................regressor__alpha=2.0; total time=   0.1s
[CV] END ..............................regressor__alpha=0.01; total time=   0.1s
[CV] END ...............................regressor__alpha=2.0; total time=   0.1s[CV] END ...............................regressor__alpha=5.0; total time=   0.0s

[CV] END ...................

In [12]:
# Get the best-performing model
improved_regressor = grid.best_estimator_["regressor"]

print(f"estimator : {improved_regressor}", end="\n\n")

# Evaluate the model on training and test datasets
r2_train = r2_score(y_train["Output"], improved_regressor.predict(X_train))
r2_test = r2_score(y_test["Output"], improved_regressor.predict(X_test))

print(f"r2 (train) = {r2_train}")
print(f"r2 (val)   = {grid.best_score_}")
print(f"r2 (test)  = {r2_test}")

estimator : Lasso(alpha=0.1)

r2 (train) = 0.5299400684465791
r2 (val)   = 0.4796890029011515
r2 (test)  = 0.4765049482890068


In [13]:
# Sort features by their significance (the absolute value of their coefficient)
sorted_features = np.argsort(np.abs(improved_regressor.coef_))[::-1]

# List 10 most significant features
print("most significant features:")
for feature in sorted_features[:10]:
    print(
        f"- {feature_names[feature]}, coef={improved_regressor.coef_[feature]}"
    )

# Get features with nonzero weights
nonzero_features = improved_regressor.coef_.nonzero()[0]

print(f"\nthere are {len(nonzero_features)} features with nonzero weights:")
print(*feature_names[nonzero_features])

# Plot the model's weights
fig = px.bar(
    x=feature_names,
    y=improved_regressor.coef_,
    title="Significance of each feature in the improved model",
)
fig.update_layout(xaxis_title="feature", yaxis_title="coefficient")

most significant features:
- Input83, coef=0.8839305582058237
- Input223, coef=0.8631975877278412
- Input167, coef=0.7024663304841298
- Input193, coef=0.6322905338850379
- Input292, coef=0.6039716620564994
- Input342, coef=0.588204608556254
- Input184, coef=0.5662672671827017
- Input136, coef=0.5262986483506887
- Input173, coef=0.4982080994512175
- Input18, coef=0.3799393659710356

there are 59 features with nonzero weights:
Input18 Input25 Input36 Input44 Input53 Input59 Input62 Input65 Input69 Input78 Input83 Input88 Input93 Input94 Input98 Input100 Input121 Input123 Input132 Input136 Input138 Input153 Input160 Input167 Input170 Input172 Input173 Input180 Input184 Input191 Input193 Input194 Input203 Input204 Input213 Input217 Input223 Input226 Input231 Input232 Input235 Input236 Input241 Input248 Input250 Input281 Input286 Input292 Input298 Input300 Input304 Input335 Input342 Input362 Input363 Input381 Input387 Input389 Input394


|         | relative improvement over the baseline model ($R^2$) |
|---------|------------------------------------------------------|
| train   | $\color{red} -17\%$                                  |
| val     | $\color{green} +46\%$                                |
| test    | $\color{green} +49\%$                                |

---

In [14]:
path = Path("validation_data.csv")

if path.exists():
    # Load the dataset into a DataFrame
    df = pd.read_csv(path, sep=";")

    # Extract input (X) and output (y) variables
    X = df.filter(regex="^Input").to_numpy()
    y = df[["Class", "Output"]].to_numpy()

    # Aggregate output variables by their type
    y = {"Class": y[:, 0], "Output": y[:, 1]}

    # Evaluate each model on the entire dataset
    accuracy_base = accuracy_score(y["Class"], base_classifier.predict(X))
    accuracy_improved = accuracy_score(
        y["Class"], improved_classifier.predict(X)
    )
    r2_base = r2_score(y["Output"], base_regressor.predict(X))
    r2_improved = r2_score(y["Output"], improved_regressor.predict(X))

    print(f"accuracy (base)     = {accuracy_base}")
    print(f"accuracy (improved) = {accuracy_improved}", end="\n\n")
    print(f"r2 (base)     = {r2_base}")
    print(f"r2 (improved) = {r2_improved}")